[![Buka di Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/YusufArrayyan/Informatics-Practiticioner-Lecturer-Series/blob/main/Pertemuan_6_Deforestasi_Satelit_AI.ipynb)

# Pertemuan 6 — Attributing Direct Drivers to Deforestation Alerts with Satellite Imagery and AI
**Studi Kasus:** Sistem Atribusi Pemicu Langsung Deforestasi untuk Balai Penegakan Hukum Kehutanan di Bengkulu

**Identitas**
- Nama: Muhammad Yusuf Arrayyan
- NPM: G1A024009
- Program Studi: S1 Teknik Informatika, Universitas Bengkulu (UNIB)
- Kegiatan: Informatics Practitioner Lecture Series
- Narasumber: Dr. Bart Slagter (Laboratory of Geo-Information Science and Remote Sensing, Wageningen University & Research)
- Waktu: Senin, 19 Oktober 2026, 14.00 WIB

> Data pada notebook ini **sintetis** (rentang fitur dirancang dari karakteristik umum tiap pemicu). Akurasi tidak mencerminkan performa pada citra nyata, dan lokasi alert pada bagian akhir adalah simulasi.

## Ringkasan Materi
1. Sistem peringatan dini seperti **RADD** (radar Sentinel-1) dan **Global Forest Watch** menjawab pertanyaan **"di mana dan kapan"** hutan hilang, tetapi belum **"mengapa"**. **Atribusi pemicu langsung** (sawit industri, pertanian rakyat, tambang, jalan penebangan) menentukan respons yang tepat.
2. **Fusi multi-sensor:** optik Sentinel-2 (indeks NDVI dan NBR) kaya informasi tetapi terhalang awan; radar Sentinel-1 (backscatter VH, rasio VH/VV) menembus awan dan memberi informasi struktur.
3. **Fitur tambahan:** morfologi poligon (luas, elongasi/bentuk memanjang) dan **deret waktu pascagangguan** (kecepatan pemulihan vegetasi).
4. Metode: ML klasik (Random Forest) hingga *deep learning* (CNN, U-Net, Transformer); tantangan utama **class imbalance** dan kebutuhan data latih beranotasi ahli.
5. Arah lanjutan: *Edge AI* untuk drone dan gawai lapangan, serta antarmuka yang ramah petugas.

## Latar Belakang
Hutan di Bengkulu dan sekitarnya mengalami tekanan dari ekspansi perkebunan, pembukaan lahan masyarakat, tambang, dan pembalakan. Sistem satelit menghasilkan banyak peringatan kehilangan tutupan, tetapi petugas lapangan terbatas. Tanpa mengetahui **apa pemicunya**, patroli sulit diprioritaskan: bukaan pertanian rakyat berskala kecil tentu ditangani berbeda dengan bukaan tambang atau jalan penebangan baru di dalam hutan lindung, yang menandakan potensi kegiatan ilegal.

Studi kasus ini membangun prototipe **model atribusi pemicu** yang mengubah setiap alert menjadi label pemicu beserta tingkat keyakinan, lalu menerjemahkannya menjadi **rekomendasi prioritas patroli**. Studi kasus juga menguji dua pertanyaan dari materi: mengapa fusi radar-optik diperlukan di wilayah tropis berawan, dan apa dampak ketidakseimbangan kelas.

## Rumusan Masalah
1. Fitur apa yang membedakan keempat pemicu deforestasi?
2. Seberapa akurat model dapat mengatribusikan pemicu dari fitur gabungan?
3. Bagaimana kinerja ketika sebagian citra optik hilang karena awan, dan apa peran radar?
4. Bagaimana kelas langka (tambang) memengaruhi model dan metrik yang perlu dipantau?
5. Bagaimana prediksi model diterjemahkan menjadi prioritas tindakan secara bertanggung jawab?

## Tujuan
1. Membangun dataset sintetis alert dengan tujuh fitur multi-sensor, morfologi, dan deret waktu.
2. Melatih dan mengevaluasi Random Forest (presisi, recall, F1, matriks konfusi, kontribusi fitur).
3. Menjalankan **ablasi fitur** dengan 40% data optik hilang.
4. Mengukur dampak class imbalance dan penanganannya.
5. Membuat aturan keputusan berbasis probabilitas untuk prioritas patroli.

## Dasar Teori
- **NDVI** = (NIR − Red)/(NIR + Red): ukuran kehijauan vegetasi; **NBR** = (NIR − SWIR)/(NIR + SWIR): sensitif terhadap lahan terbuka dan bekas terbakar. Yang dipakai adalah penurunannya (*drop*) sebelum dan sesudah kejadian.
- **Radar SAR (Sentinel-1):** backscatter VH (dB) sensitif terhadap struktur kanopi; rasio VH/VV membantu membedakan permukaan. Radar aktif sehingga bekerja menembus awan dan malam hari.
- **Karakter pemicu (asumsi dalam data sintetis):** sawit industri → bukaan luas dengan nilai `Elongasi_Shape` tinggi; pertanian rakyat → kecil, vegetasi pulih cepat; tambang → penurunan NDVI/NBR terdalam dan hampir tidak pulih; jalan penebangan → bukaan sempit dengan nilai `Elongasi_Shape` terendah (skala fitur ini ditentukan di kode).
- **Random Forest** untuk klasifikasi multikelas; **F1 makro** merata-ratakan F1 tiap kelas (adil terhadap kelas langka); **recall** kelas langka mengukur seberapa banyak kasus penting yang tertangkap.
- **Class imbalance:** kelas langka cenderung diabaikan model; penanganan umum: `class_weight`, *resampling*, atau pengumpulan data tambahan.

## Dataset
1.000 alert sintetis (250 per pemicu), tujuh fitur:
| Kelompok | Fitur |
|---|---|
| Morfologi | `Luas_Hektare`, `Elongasi_Shape` |
| Optik | `NDVI_Drop`, `NBR_Drop` |
| Radar | `SAR_VH_dB`, `SAR_Ratio_VHVV` |
| Deret waktu | `NDVI_Pulih_90hr` |

Nilai tiap kelas diambil dari rentang yang dirancang, lalu ditambah **derau pengukuran** agar kelas saling tumpang tindih. Kelas `Tambang` hanya berarti "bukaan tambang terdeteksi"; legal atau ilegalnya harus dicek pada data izin (overlay IUP).

## Metodologi
1. Bangun dataset → 2. Latih Random Forest (split 80/20 berstrata) dan evaluasi → 3. Ablasi: 40% poligon kehilangan data optik (diimputasi median), bandingkan optik saja, radar saja, fusi, dan fusi lengkap → 4. Class imbalance: kelas Tambang diperkecil menjadi 24 sampel latih, bandingkan tanpa dan dengan `class_weight='balanced'` → 5. Matriks konfusi dan kontribusi fitur → 6. Inferensi tiga alert simulasi dengan aturan prioritas.

## Implementasi
Setiap kelompok keluaran diikuti sel **Interpretasi**.

In [ ]:
import os
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestClassifier
from sklearn.impute import SimpleImputer
from sklearn.pipeline import make_pipeline
from sklearn.metrics import classification_report, confusion_matrix, recall_score, f1_score

os.makedirs("data", exist_ok=True)
os.makedirs("output", exist_ok=True)
plt.rcParams.update({"axes.spines.top": False, "axes.spines.right": False, "figure.dpi": 110})
np.random.seed(42)
rng = np.random.default_rng(42)

## 1. Dataset sintetis peringatan deforestasi
Fitur: luas (ha), elongasi bentuk, penurunan NDVI & NBR (optik), backscatter VH & rasio VH/VV (radar),
dan **pemulihan NDVI 90 hari** (fitur deret waktu).
Catatan: kelas `Tambang` hanya menyatakan *bukaan tambang terdeteksi*; status legal/ilegal perlu dicek ke data izin (overlay IUP).

In [ ]:
n = 250
def kelas(nama, luas, elong, ndvi, nbr, vh, ratio, pulih):
    u = lambda lo_hi: rng.uniform(lo_hi[0], lo_hi[1], n)
    return pd.DataFrame({"Luas_Hektare": u(luas), "Elongasi_Shape": u(elong), "NDVI_Drop": u(ndvi), "NBR_Drop": u(nbr),
                         "SAR_VH_dB": u(vh), "SAR_Ratio_VHVV": u(ratio), "NDVI_Pulih_90hr": u(pulih), "Driver": nama})

df = pd.concat([
    kelas("Sawit_Industri",   (15, 85),  (0.70, 0.95), (0.35, 0.60), (0.30, 0.55), (-17.5, -15.0), (0.35, 0.50), (0.10, 0.30)),
    kelas("Pertanian_Rakyat", (0.5, 3.2), (0.40, 0.75), (0.20, 0.45), (0.15, 0.40), (-15.5, -13.0), (0.45, 0.60), (0.25, 0.55)),
    kelas("Tambang",          (3, 25),   (0.50, 0.85), (0.65, 0.90), (0.60, 0.85), (-22.0, -18.5), (0.20, 0.35), (0.00, 0.08)),
    kelas("Jalan_Penebangan", (1.5, 12), (0.05, 0.28), (0.40, 0.70), (0.35, 0.65), (-18.0, -14.5), (0.30, 0.48), (0.05, 0.25)),
], ignore_index=True)

# Derau pengukuran sensor agar kelas saling tumpang-tindih (lebih realistis)
SIGMA = {"Luas_Hektare": 0.12, "Elongasi_Shape": 0.08, "NDVI_Drop": 0.08, "NBR_Drop": 0.08,
         "SAR_VH_dB": 1.4, "SAR_Ratio_VHVV": 0.05, "NDVI_Pulih_90hr": 0.07}
for kol, s in SIGMA.items():
    base = df[kol].copy()
    df[kol] = base * (1 + rng.normal(0, s, len(df))) if kol == "Luas_Hektare" else base + rng.normal(0, s, len(df))
df["Luas_Hektare"] = df["Luas_Hektare"].clip(lower=0.2)
df["Elongasi_Shape"] = df["Elongasi_Shape"].clip(0.01, 1.0)
df.round(4).to_csv("data/peringatan_deforestasi_sintetis.csv", index=False)
print(df["Driver"].value_counts().to_string())
print(df.groupby("Driver").mean(numeric_only=True).round(2).T)

### Interpretasi: karakter tiap pemicu
- Dataset seimbang: 250 alert per pemicu.
- Rata-rata fitur sesuai karakter yang dirancang. **Sawit industri**: luas ±49,8 ha dan elongasi tinggi (0,82). **Pertanian rakyat**: luas ±1,9 ha dan pemulihan NDVI tertinggi (0,40). **Tambang**: penurunan NDVI terdalam (0,77), backscatter VH terendah (−20,4 dB), dan pemulihan nyaris nol (0,04). **Jalan penebangan**: `Elongasi_Shape` terendah (0,17) dengan luas ±6,7 ha.
- Tidak ada satu fitur yang memisahkan semua kelas; pembeda berbeda untuk tiap pasangan, sehingga fitur perlu digabungkan.

## 2. Pelatihan model atribusi (Random Forest) & evaluasi

In [ ]:
FITUR_OPTIK = ["NDVI_Drop", "NBR_Drop"]
FITUR_SAR = ["SAR_VH_dB", "SAR_Ratio_VHVV"]
FITUR_MORFO = ["Luas_Hektare", "Elongasi_Shape"]
FITUR_WAKTU = ["NDVI_Pulih_90hr"]
FITUR = FITUR_MORFO + FITUR_OPTIK + FITUR_SAR + FITUR_WAKTU

X, y = df[FITUR], df["Driver"]
X_tr, X_te, y_tr, y_te = train_test_split(X, y, test_size=0.20, random_state=42, stratify=y)

rf = RandomForestClassifier(n_estimators=150, max_depth=8, random_state=42, n_jobs=-1).fit(X_tr, y_tr)
y_pred = rf.predict(X_te)
print(classification_report(y_te, y_pred, digits=3))

imp = pd.Series(rf.feature_importances_, index=FITUR).sort_values(ascending=False)
print("Kontribusi fitur (%):")
print((imp * 100).round(1).to_string())

### Interpretasi: evaluasi model atribusi
- Akurasi pada 200 data uji **99,5%** (hanya 1 kesalahan: satu jalan penebangan terprediksi pertanian rakyat); F1 makro 0,995.
- Fitur paling berkontribusi: **Luas_Hektare 30,2%** dan **Elongasi_Shape 25,9%**, disusul NBR_Drop 10,6%, SAR_VH_dB 10,1%, NDVI_Drop 8,7%, NDVI_Pulih_90hr 8,4%, dan SAR_Ratio_VHVV 6,2%. Bentuk dan ukuran poligon paling membedakan pada data ini.
- **Peringatan:** akurasi hampir sempurna terjadi karena data sintetis dirancang mudah dipisahkan. Pada citra nyata, kelas saling tumpang tindih jauh lebih besar dan angka sebesar ini tidak boleh diharapkan.

## 3. Mengapa fusi multi-sensor? Ablasi dengan tutupan awan
Di hutan tropis, sebagian citra optik tertutup awan. Kita simulasikan **40% poligon** tanpa data optik (NaN),
lalu bandingkan: optik saja, radar saja, dan fusi optik + radar (+ morfologi + deret waktu).
Nilai optik yang hilang diisi median (imputasi sederhana).

In [ ]:
X_awan = X.copy()
mask_awan = rng.random(len(X_awan)) < 0.40
X_awan.loc[mask_awan, FITUR_OPTIK] = np.nan
Xa_tr, Xa_te, ya_tr, ya_te = train_test_split(X_awan, y, test_size=0.20, random_state=42, stratify=y)

skenario = {
    "Optik saja (NDVI, NBR)":                FITUR_OPTIK,
    "Radar saja (VH, VH/VV)":                FITUR_SAR,
    "Fusi optik + radar":                    FITUR_OPTIK + FITUR_SAR,
    "Fusi + morfologi + deret waktu":        FITUR,
}
hasil = []
for nama, kolom in skenario.items():
    m = make_pipeline(SimpleImputer(strategy="median"),
                      RandomForestClassifier(n_estimators=150, max_depth=8, random_state=42, n_jobs=-1))
    m.fit(Xa_tr[kolom], ya_tr)
    p = m.predict(Xa_te[kolom])
    hasil.append((nama, f1_score(ya_te, p, average="macro")))
abl = pd.DataFrame(hasil, columns=["fitur", "F1_macro"]).round(3)
print(abl.to_string(index=False))

print("Catatan: skor 1.0 pada baris terakhir terjadi karena fitur morfologi (luas, elongasi) memisahkan kelas hampir sempurna "
      "pada data sintetis. Pada data nyata, bentuk poligon saja jarang cukup; yang ingin ditunjukkan di sini adalah "
      "radar menjaga performa saat optik hilang.")
fig, ax = plt.subplots(figsize=(7, 3.4))
ax.barh(abl["fitur"], abl["F1_macro"], color=["#98A2B3", "#98A2B3", "#1570EF", "#1570EF"], height=0.55)
ax.invert_yaxis(); ax.set_xlim(0, 1)
for i, v in enumerate(abl["F1_macro"]):
    ax.text(v + 0.01, i, f"{v:.2f}", va="center", fontweight="bold")
ax.set_xlabel("F1 macro (dengan 40% poligon tertutup awan)")
ax.set_title("Nilai tambah fusi multi-sensor")
plt.tight_layout()
plt.savefig("output/06_ablasi_fusi_sensor.png", dpi=200)
plt.show()

### Interpretasi: ablasi fusi multi-sensor (40% optik tertutup awan)
- F1 makro: **optik saja 0,539**, **radar saja 0,705**, **fusi optik + radar 0,717**, **fusi + morfologi + deret waktu 1,000**.
- Saat awan menutup sebagian besar citra optik, **optik saja turun paling jauh**; radar tetap stabil karena tidak terpengaruh awan, dan menggabungkannya dengan optik memberi sedikit tambahan. Inilah alasan fusi multi-sensor penting di hutan tropis.
- Skor 1,000 pada baris terakhir terjadi karena fitur morfologi memisahkan kelas hampir sempurna pada data sintetis. Pesan yang ingin ditunjukkan adalah **radar menjaga kinerja saat optik hilang**, bukan bahwa 1,000 dapat dicapai pada data nyata.

## 4. Tantangan class imbalance
Di lapangan, tambang liar jauh lebih jarang daripada ladang kecil. Kita buat data latih dengan kelas `Tambang`
hanya ±8% dan bandingkan **recall** kelas itu antara model biasa dan `class_weight="balanced"`.

In [ ]:
tr = pd.concat([X_tr, y_tr], axis=1)
tambang_idx = tr[tr["Driver"] == "Tambang"].sample(frac=0.12, random_state=1).index
tr_imb = pd.concat([tr[tr["Driver"] != "Tambang"], tr.loc[tambang_idx]])
print("Distribusi kelas latih (tidak seimbang):")
print(tr_imb["Driver"].value_counts().to_string())

baris = []
for nama, cw in [("Tanpa penyeimbangan", None), ("class_weight='balanced'", "balanced")]:
    m = RandomForestClassifier(n_estimators=150, max_depth=8, class_weight=cw, random_state=42, n_jobs=-1)
    m.fit(tr_imb[FITUR], tr_imb["Driver"])
    p = m.predict(X_te)
    baris.append((nama, recall_score(y_te, p, labels=["Tambang"], average=None)[0], f1_score(y_te, p, average="macro")))
imb = pd.DataFrame(baris, columns=["strategi", "recall_Tambang", "F1_macro"]).round(3)
print(imb.to_string(index=False))
print("\nRecall kelas langka krusial: tambang yang terlewat berarti perambahan tidak tertangani.")
print("Catatan: pada data sintetis yang mudah dipisahkan ini dampak ketidakseimbangan kecil, dan pembobotan bisa "
      "menukar sedikit presisi dengan recall. Pada citra nyata (kelas tumpang-tindih) dampaknya biasanya jauh lebih besar.")

### Interpretasi: class imbalance
- Data latih sengaja dibuat tidak seimbang: 200 sampel untuk tiga kelas dan hanya **24 untuk Tambang**.
- Tanpa penyeimbangan: recall Tambang **0,98**, F1 makro 0,995. Dengan `class_weight='balanced'`: recall Tambang **0,92**, F1 makro 0,980.
- Pada data sintetis yang mudah dipisahkan, ketidakseimbangan hampir tidak merugikan, dan pembobotan justru menukar sedikit presisi (bahkan sedikit menurunkan recall) di sini. Hasil ini dilaporkan apa adanya.
- Pada citra nyata dengan kelas tumpang tindih, kelas langka biasanya jauh lebih terabaikan, sehingga **recall kelas langka harus selalu dipantau secara terpisah**, bukan hanya akurasi total (akurasi tinggi bisa menyembunyikan kelas penting yang terlewat).

## 5. Matriks konfusi & kontribusi fitur

In [ ]:
labels = sorted(y.unique())
cm = confusion_matrix(y_te, y_pred, labels=labels)

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 5.5))
cax = ax1.matshow(cm, cmap="Blues", alpha=0.85)
fig.colorbar(cax, ax=ax1, fraction=0.046, pad=0.04)
ax1.set_xticks(range(len(labels))); ax1.set_yticks(range(len(labels)))
ax1.set_xticklabels(labels, rotation=35, ha="left", fontsize=9, fontweight="bold")
ax1.set_yticklabels(labels, fontsize=9, fontweight="bold")
ax1.set_title("Matriks konfusi atribusi pemicu", fontsize=12, fontweight="bold", pad=20)
ax1.set_xlabel("Prediksi model", fontweight="bold", labelpad=10); ax1.set_ylabel("Kondisi aktual", fontweight="bold")
for i in range(len(labels)):
    for j in range(len(labels)):
        ax1.text(j, i, str(cm[i, j]), ha="center", va="center",
                 color="black" if cm[i, j] < cm.max() / 2 else "white", fontweight="bold")

ax2.barh(imp.index, imp.values * 100, color="#1570EF", height=0.55)
ax2.invert_yaxis()
ax2.set_title("Kontribusi fitur (%)", fontsize=12, fontweight="bold", pad=15)
ax2.set_xlabel("Kepentingan fitur (%)", fontweight="bold")
for i, v in enumerate(imp.values * 100):
    ax2.text(v + 0.3, i, f"{v:.1f}%", va="center", fontsize=9, fontweight="bold")
plt.tight_layout()
plt.savefig("output/06_evaluasi_model.png", dpi=200, bbox_inches="tight")
plt.show()

## 6. Inferensi pada peringatan baru (SIMULASI) + prioritas patroli
Rekomendasi memakai **probabilitas** dan ambang keyakinan: prediksi dengan keyakinan rendah diarahkan ke verifikasi
citra resolusi tinggi, bukan langsung ke tindakan lapangan.

In [ ]:
alert_baru = pd.DataFrame([
    # Luas, Elongasi, NDVI_Drop, NBR_Drop, VH, VH/VV, NDVI_Pulih
    [42.5, 0.88, 0.48, 0.42, -16.2, 0.41, 0.18],
    [3.2, 0.12, 0.55, 0.49, -16.8, 0.38, 0.12],
    [9.1, 0.65, 0.78, 0.72, -20.5, 0.26, 0.03],
], columns=FITUR_MORFO + FITUR_OPTIK + FITUR_SAR + FITUR_WAKTU)[FITUR]
lokasi = ["Alert simulasi A (penyangga hutan lindung)", "Alert simulasi B (koridor di dalam hutan)", "Alert simulasi C (bantaran sungai)"]

prob = rf.predict_proba(alert_baru)
pred = rf.classes_[prob.argmax(axis=1)]
PRIORITAS_1 = {"Tambang", "Jalan_Penebangan"}
for i in range(len(alert_baru)):
    yakin = prob[i].max() * 100
    print(f"{lokasi[i]}")
    print(f"  -> Pemicu terdeteksi : {pred[i]}  (keyakinan {yakin:.1f}%)")
    if yakin < 60:
        aksi = "VERIFIKASI: keyakinan rendah, cek citra resolusi tinggi sebelum menerjunkan tim"
    elif pred[i] in PRIORITAS_1:
        aksi = "PRIORITAS 1: koordinasikan tim penegakan hukum untuk verifikasi lapangan (cek izin/IUP)"
    else:
        aksi = "PRIORITAS 2: pantau dengan citra resolusi tinggi"
    print(f"  -> Rekomendasi       : {aksi}\n")

### Interpretasi: dari prediksi ke prioritas patroli
- **Alert A (penyangga hutan lindung):** Sawit_Industri (keyakinan 99,9%) → Prioritas 2: pantau dengan citra resolusi tinggi.
- **Alert B (koridor di dalam hutan):** Jalan_Penebangan (100%) → **Prioritas 1**: koordinasi penegakan hukum untuk verifikasi lapangan (cek izin).
- **Alert C (bantaran sungai):** Tambang (100%) → **Prioritas 1**: verifikasi lapangan dan cek izin IUP.
- Aturan keputusan memakai **probabilitas**: bila keyakinan <60%, alert diarahkan ke verifikasi citra resolusi tinggi sebelum menerjunkan tim. Prediksi model adalah **petunjuk prioritas, bukan bukti hukum**; tindakan hukum memerlukan verifikasi lapangan dan data perizinan.
- Keyakinan sangat tinggi (99,9–100%) di sini mencerminkan data sintetis yang mudah; model nyata cenderung kurang yakin dan memerlukan kalibrasi probabilitas.

## Ringkasan
- Atribusi pemicu mengubah peringatan "di mana & kapan" menjadi informasi "mengapa", sehingga patroli lebih terarah.
- Radar menjaga performa saat optik tertutup awan; fusi dan fitur deret waktu menambah daya pembeda.
- Class imbalance perlu ditangani (bobot kelas, resampling) dan dievaluasi lewat recall kelas langka.
- Prediksi model = **petunjuk prioritas**, bukan bukti hukum: butuh verifikasi lapangan dan data perizinan.
- Arah riset lanjutan dari materi: Edge AI untuk drone/gawai, CNN/U-Net/Transformer pada citra asli, serta dasbor
  yang ramah polisi hutan (lihat Pertemuan 5).

## Hasil dan Pembahasan (ringkasan)
| Analisis | Temuan |
|---|---|
| Model atribusi | Akurasi 99,5% (data sintetis); fitur teratas luas (30,2%) dan elongasi (25,9%) |
| Ablasi 40% tertutup awan | F1 makro: optik 0,54; radar 0,71; fusi 0,72; fusi lengkap 1,00 |
| Class imbalance | Recall Tambang 0,98 (tanpa penyeimbangan) vs 0,92 (class_weight balanced) |
| Prioritas patroli | Tambang dan jalan penebangan → Prioritas 1; sawit → Prioritas 2; keyakinan <60% → verifikasi |

## Kesimpulan
1. Atribusi pemicu mengubah peringatan "di mana dan kapan" menjadi informasi "mengapa", sehingga patroli lebih terarah.
2. Radar menjaga kinerja ketika optik tertutup awan; fusi dan fitur morfologi serta deret waktu menambah daya pembeda.
3. Class imbalance perlu dievaluasi lewat recall kelas langka, bukan hanya akurasi total.
4. Keluaran model adalah petunjuk prioritas yang harus diverifikasi lapangan dan dicocokkan dengan data perizinan, bukan bukti hukum.

## Keterbatasan
- Seluruh data sintetis dan mudah dipisahkan, sehingga akurasi mendekati sempurna; tidak mencerminkan kinerja pada citra nyata.
- Model tabular (ringkasan fitur per poligon), bukan CNN/U-Net atas citra; informasi spasial dan konteks lanskap tidak dimanfaatkan.
- Tidak ada validasi spasial (uji pada wilayah berbeda) maupun kalibrasi probabilitas.
- Label `Tambang` tidak membedakan legal dan ilegal.

## Saran Pengembangan
- Menggunakan data nyata beranotasi seperti **ForestNet** dan alert RADD/GLAD melalui Google Earth Engine; validasi dengan survei lapangan.
- Mencoba CNN/U-Net atau Transformer pada potongan citra Sentinel-1/2 dan deret waktu panjang.
- Menambahkan overlay izin (IUP, HGU), batas kawasan hutan, dan jaringan jalan sebagai fitur kontekstual.
- Menyiapkan antarmuka dasbor untuk petugas (lihat Pertemuan 5) dan eksplorasi *Edge AI* di perangkat lapangan.

## Referensi
> Rujukan di bawah adalah bacaan lanjutan. Periksa kembali penulisan, tahun, dan volume sebelum dikutip dalam laporan resmi.
- Reiche, J., dkk. (2021). Combining satellite data for better tropical forest monitoring (RADD alerts). *Nature Climate Change*, 11, 1–3.
- Hansen, M. C., dkk. (2013). High-resolution global maps of 21st-century forest cover change. *Science*, 342, 850–853.
- Irvin, J., dkk. (2020). ForestNet: Classifying drivers of deforestation in Indonesia using deep learning on satellite imagery. *NeurIPS Workshop on Tackling Climate Change with ML*.
- Global Forest Watch (globalforestwatch.org).
- Breiman, L. (2001). Random Forests. *Machine Learning*, 45(1), 5–32.
- Materi Practitioner Lecture Series, Pertemuan 6 (dokumen ringkasan).